In [15]:
from ..Agent import *

class employee(BaseModel):
    name:str=Field(default=None,description='员工姓名')
    score:str=Field(default=None,description='员工绩效')
    gene:Literal['男','女']=Field(default='男',description='员工性别')
    isSend:bool=Field(default=False,description='是否发放奖金')
    money:float=Field(default=0.0,description='奖金金额')

class event(BaseModel):
    date: str = Field(None, description="事件发生的日期，例如'2025-01-01'")
    name: str = Field(None, description="事件的名称或标题，例如'董事会会议'")

class event_detail(BaseModel):
    person: str = Field(None, description="事件参与人，例如'张三'")
    content: str = Field(None, description="事件的具体内容，例如'去超市买菜'")

@tool(description='根据员工名查看员工绩效级别')
def emp_score(emp_name:str) ->str:
    score=None
    if(emp_name=='张三'):
        score='A'
    elif(emp_name=='李四'):
       score='B'
    elif(emp_name=='王五'):
        score='C'
    if score is None:
        return f'不存在名为{emp_name}的员工'
    return f'{emp_name}绩效为{score}'
@tool(parse_docstring=True)
def score2money(score:str)->float:
    '''
    根据绩效来发奖金

    Args:
        score:绩效级别

    Returns:
         float:发放的奖金金额
    '''
    if(score=='A'):
        return 3000.0
    elif(score=='B'):
        return 2000.0
    elif(score=='C'):
        return 1000.0
    else:
        return 0.0

load_dotenv(override=True)
#必须关闭思考模式，deepseek默认为思考模式，思考模式不支持结构化输出
#因为deepseek底层使用tool_choice作为伪工具传递结构化输出，思考模式不支持tool_choice
DEEPSEEK_API_KEY=os.getenv('DEEPSEEK_API_KEY')
DEEPSEEK_BASE_URL=os.getenv('DEEPSEEK_BASE_URL')
model=init_chat_model(
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    model='deepseek-v4-flash',
    model_provider='deepseek',
    extra_body={"thinking":{"type":"disabled"}}
)

In [36]:
from langchain.agents.structured_output import ToolStrategy

#修改response_format参数用于结构化输出
#ProviderStrategy是模型服务商提供的(有些模型不支持)，ToolStrategy(是langchain提供的，兼容大部分)
#AutoStrategy自动在前两种选择
myagent=create_agent(
    model=model,
    tools=[emp_score,score2money],
    response_format=ToolStrategy(employee),
    system_prompt='请分析员工当前情况：'
                  '1.查询员工绩效'
                  '2.根据绩效查询应当发放的奖金'
                  '3.发放奖金，将状态返回为已发放'
                  '4.返回结构化输出报告'
)
messages=HumanMessage('请分析员工张三，性别男')
response=myagent.invoke({
    'messages':messages
})
rprint(response['structured_response'])

employee(name='张三', score='A', gene='男', isSend=True, money=3000.0)

In [17]:
#使用Union可以进行多结构化输出，模型在多个结构化输出中选取最合适的那一个
myagent2=create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[event_detail,event],
        # envent_detail
    )
)
messages=HumanMessage('根据已有信息选择合适的结构化输出，从这段话抽取结构化信息：2026年高考人数突破100w')
response=myagent2.invoke({
    'messages':messages
})
rprint(response['structured_response'])

event(date='2026', name='高考人数突破100w')